# NB 15. GUARDAR SEGMENTACIONES DE VIDEO EN OTRO ARCHIVO

## 0. Instalar paquetes, importar librerias y archivos

In [ ]:
!pip install supervision ultralytics -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.6/251.6 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 32.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 3.3 MB/s eta 0:00:00


In [ ]:
import cv2
import numpy as np
import supervision as sv
from google.colab import drive
import pandas as pd

drive.mount("/content/drive")

Mounted at /content/drive


## 1. Crear funciones para guardar las segmentaciones

In [ ]:
from pycocotools import mask as mask_utils
import json
import os
import numpy as np

In [ ]:
def guardar_detecciones_frame(indice_frame:int,detecciones:sv.Detections,ruta_salida:str):
    """
    Guardar las mascaras de un frame especifico en formato RLE
    + los metadatos. Directamente se la pedi a Claude :(

    """

    if len(detecciones)==0:
        datos_rle = [] #Guardar lista vacia para mantener indexado

    else:
        datos_rle=[]


        for i in range(len(detecciones)):
            mascara = detecciones.mask[i]

            #RLE ocupa un array en orden Fortran y uint8
            rle = mask_utils.encode(
                np.asfortranarray(mascara.astype(np.uint8))
            )

            #counts viene como bytes, asi que hay que decodificar para JSON
            rle["counts"] = rle["counts"].decode("utf-8")

            obj = {
                "rle":rle, # #{"size": [H, W], "counts": "..."}
                "xyxy":detecciones.xyxy[i].tolist(),
            }

            #Campos opcionales, solo se guardan si existen
            if detecciones.confidence is not None:
                obj["confidence"] = float(detecciones.confidence[i])
            if detecciones.class_id is not None:
                obj["class_id"] = int(detecciones.class_id[i])
            if detecciones.tracker_id is not None:
                obj["tracker_id"] = int(detecciones.tracker_id[i])

            if detecciones.data:
                obj["data"] = {
                    clave: valores[i].tolist() if isinstance(valores,np.ndarray) else valores[i]
                    for clave,valores in detecciones.data.items()
                }

            datos_rle.append(obj)

        RUTA_SALIDA = os.path.join(ruta_salida,f"frame_{indice_frame:08d}.json")
        with open(RUTA_SALIDA,"w") as f:
            json.dump(datos_rle,f)


#PARA CARGAR LOS DATOS NUEVAMENTE
def cargar_detecciones_frame(indice_frame:int,ruta_carpeta:str):
    """
    Reconstruye sv.Detections desde el JSON guardado. Igual me la dio Claude
    """
    in_path = os.path.join(ruta_carpeta,f"frame_{indice_frame:08d}.json")
    with open(in_path) as f:
        datos_rle = json.load(f)

    if len(datos_rle) ==0:
        return sv.Detections.empty()
    mascaras,cajas,confianza,clase,tracker_ids = [],[],[],[],[]

    for objeto in datos_rle:
        rle = objeto["rle"]
        rle["counts"] = rle["counts"].encode("utf-8") #de vuelta a bytes
        mascaras.append(mask_utils.decode(rle).astype(bool))
        cajas.append(objeto["xyxy"])
        confianza.append(objeto.get("confidence"))
        clase.append(objeto.get("class_id"))
        tracker_ids.append(objeto.get("tracker_id"))

    return sv.Detections(
        xyxy = np.array(cajas,dtype=np.float32),
        mask=np.stack(mascaras),
        confidence = np.array(confianza,dtype=np.float32) if confianza[0] is not None else None,
        class_id=np.array(clase, dtype=int) if clase[0] is not None else None,
        tracker_id=np.array(tracker_ids, dtype=int) if tracker_ids[0] is not None else None,
    )






## 2. Ejecutar segmentacion en 5 frames de un video

In [ ]:
from ultralytics.models.sam import SAM3VideoSemanticPredictor
import torch


rutaVideo = "/content/drive/MyDrive/PROYECTO SAM 3/VIDEOS/IMG_9938.MOV"
ruta_sam = "/content/drive/MyDrive/PROYECTO SAM 3/sam3.pt"


#IMPRIMIR INFORMACION DEL VIDEO
info = sv.VideoInfo.from_video_path(rutaVideo)

print(f"Resolucion: {info.width} x {info.height}")
print(f"FPS: {info.fps}")
print(f"Frames totales: {info.total_frames}")


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Resolucion: 1080 x 1920
FPS: 29.988256260752216
Frames totales: 19407


In [ ]:
#CREAR EL PREDICTOR SEMANTICO DE VIDEO
configuraciones = dict(
    conf = 0.25,
    task = "segment",
    mode = "predict",
    model=ruta_sam
)

if torch.cuda.is_available():
    configuraciones["half"]=True
    configuraciones["device"]="cuda"

predictor_video = SAM3VideoSemanticPredictor(overrides=configuraciones)

texto = ["robot"]

resultados_video = predictor_video(
    source=rutaVideo,
    text = texto,
    stream=True
)


In [ ]:
#CORRER EL PROCESO DE SEGMENTACION
CARPETA_SALIDA = "/content/drive/MyDrive/PROYECTO SAM 3/guardar_segmentacion/mascaras_video_completo"
numero_frame = 1
#limite_frames = 20

with torch.no_grad():
    for res in resultados_video:
        detecciones = sv.Detections.from_ultralytics(res)
        guardar_detecciones_frame(numero_frame,detecciones,CARPETA_SALIDA)


        numero_frame = numero_frame + 1
        #if(numero_frame == limite_frames ): break


Se truncaron las últimas líneas 5000 del resultado de transmisión.
video 1/1 (frame 14396/19407) /content/drive/MyDrive/PROYECTO SAM 3/VIDEOS/IMG_9938.MOV: 644x644 3 robots, 491.7ms
video 1/1 (frame 14397/19407) /content/drive/MyDrive/PROYECTO SAM 3/VIDEOS/IMG_9938.MOV: 644x644 3 robots, 462.0ms
video 1/1 (frame 14398/19407) /content/drive/MyDrive/PROYECTO SAM 3/VIDEOS/IMG_9938.MOV: 644x644 2 robots, 572.6ms
video 1/1 (frame 14399/19407) /content/drive/MyDrive/PROYECTO SAM 3/VIDEOS/IMG_9938.MOV: 644x644 3 robots, 533.1ms
video 1/1 (frame 14400/19407) /content/drive/MyDrive/PROYECTO SAM 3/VIDEOS/IMG_9938.MOV: 644x644 3 robots, 537.0ms
video 1/1 (frame 14401/19407) /content/drive/MyDrive/PROYECTO SAM 3/VIDEOS/IMG_9938.MOV: 644x644 3 robots, 545.6ms
video 1/1 (frame 14402/19407) /content/drive/MyDrive/PROYECTO SAM 3/VIDEOS/IMG_9938.MOV: 644x644 3 robots, 569.8ms
video 1/1 (frame 14403/19407) /content/drive/MyDrive/PROYECTO SAM 3/VIDEOS/IMG_9938.MOV: 644x644 2 robots, 564.6ms
video 1/1 (fr